In [1]:
print("hello")

hello


In [2]:
import sys
print(sys.version)
print(sys.executable)

3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
C:\Users\parul\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [3]:
import pandas as pd

file_path = r"C:\Users\parul\Downloads\archive\3mfanddo.csv"

df = pd.read_csv(file_path, low_memory=False)

df = df.drop(columns=["Unnamed: 0"])

df.head()

,INSTRUMENT,SYMBOL,EXPIRY_DT,STRIKE_PR,OPTION_TYP,OPEN,HIGH,LOW,CLOSE,SETTLE_PR,CONTRACTS,VAL_INLAKH,OPEN_INT,CHG_IN_OI,TIMESTAMP
0,FUTIDX,BANKNIFTY,29-Aug-2019,0.0,XX,28805.65,28924.00,28140.55,28499.30,28499.30,214569.0,1225914.96,1675780.0,234640.0,01-AUG-2019
1,FUTIDX,BANKNIFTY,26-Sep-2019,0.0,XX,28926.40,29030.55,28251.70,28611.45,28611.45,2484.0,14245.95,51400.0,-80.0,01-AUG-2019
2,FUTIDX,BANKNIFTY,31-Oct-2019,0.0,XX,29000.00,29105.00,28355.55,28699.05,28699.05,598.0,3434.43,9460.0,4860.0,01-AUG-2019
3,FUTIDX,NIFTY,29-Aug-2019,0.0,XX,11098.40,11098.40,10901.10,11015.35,11015.35,199881.0,1650955.24,19001400.0,1339200.0,01-AUG-2019
4,FUTIDX,NIFTY,26-Sep-2019,0.0,XX,11136.35,11145.20,10955.00,11066.60,11066.60,5283.0,43841.57,893625.0,66750.0,01-AUG-2019


In [4]:
import sqlite3

conn = sqlite3.connect("fno_db.sqlite")
cursor = conn.cursor()

cursor.execute("SELECT 1")
print("SQLite Working 🚀")

conn.close()

SQLite Working 🚀


In [5]:
import sqlite3

conn = sqlite3.connect("fno_db.sqlite")
cursor = conn.cursor()

# Enable foreign key support
cursor.execute("PRAGMA foreign_keys = ON;")

# -----------------------------
# Create Tables
# -----------------------------

cursor.execute("""
CREATE TABLE IF NOT EXISTS exchanges (
    exchange_id INTEGER PRIMARY KEY AUTOINCREMENT,
    exchange_name TEXT UNIQUE NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS instruments (
    instrument_id INTEGER PRIMARY KEY AUTOINCREMENT,
    symbol TEXT NOT NULL,
    instrument_type TEXT NOT NULL,
    exchange_id INTEGER NOT NULL,
    UNIQUE(symbol, instrument_type, exchange_id),
    FOREIGN KEY(exchange_id) REFERENCES exchanges(exchange_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS expiries (
    expiry_id INTEGER PRIMARY KEY AUTOINCREMENT,
    instrument_id INTEGER NOT NULL,
    expiry_dt DATE NOT NULL,
    strike_pr REAL,
    option_typ TEXT,
    UNIQUE(instrument_id, expiry_dt, strike_pr, option_typ),
    FOREIGN KEY(instrument_id) REFERENCES instruments(instrument_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS trades (
    trade_id INTEGER PRIMARY KEY AUTOINCREMENT,
    expiry_id INTEGER NOT NULL,
    trade_date DATE NOT NULL,
    open_pr REAL,
    high_pr REAL,
    low_pr REAL,
    close_pr REAL,
    settle_pr REAL,
    volume INTEGER,
    open_int INTEGER,
    chg_in_oi INTEGER,
    FOREIGN KEY(expiry_id) REFERENCES expiries(expiry_id)
);
""")

conn.commit()
print("Schema Created Successfully 🚀")

Schema Created Successfully 🚀


In [6]:
cursor.execute("INSERT OR IGNORE INTO exchanges (exchange_name) VALUES ('NSE')")
cursor.execute("INSERT OR IGNORE INTO exchanges (exchange_name) VALUES ('BSE')")
cursor.execute("INSERT OR IGNORE INTO exchanges (exchange_name) VALUES ('MCX')")

conn.commit()

cursor.execute("SELECT * FROM exchanges")
print(cursor.fetchall())

[(1, 'NSE'), (2, 'BSE'), (3, 'MCX')]


In [7]:
df.to_sql("raw_fno_data", conn, if_exists="replace", index=False)

print("Raw data inserted 🚀")

Raw data inserted 🚀


In [8]:
conn.execute("""
INSERT OR IGNORE INTO expiries (instrument_id, expiry_dt, strike_pr, option_typ)
SELECT DISTINCT
    i.instrument_id,
    r.EXPIRY_DT,
    r.STRIKE_PR,
    r.OPTION_TYP
FROM raw_fno_data r
JOIN instruments i
  ON r.SYMBOL = i.symbol
  AND r.INSTRUMENT = i.instrument_type;
""")

conn.commit()
print("Expiries inserted")

Expiries inserted


In [9]:
conn.execute("""
INSERT INTO trades (
    expiry_id,
    trade_date,
    open_pr,
    high_pr,
    low_pr,
    close_pr,
    settle_pr,
    volume,
    open_int,
    chg_in_oi
)
SELECT 
    ex.expiry_id,
    r.TIMESTAMP,
    r.OPEN,
    r.HIGH,
    r.LOW,
    r.CLOSE,
    r.SETTLE_PR,
    r.CONTRACTS,
    r.OPEN_INT,
    r.CHG_IN_OI
FROM raw_fno_data r
JOIN instruments i
  ON r.SYMBOL = i.symbol
  AND r.INSTRUMENT = i.instrument_type
JOIN expiries ex
  ON ex.instrument_id = i.instrument_id
  AND ex.expiry_dt = r.EXPIRY_DT
  AND ex.strike_pr = r.STRIKE_PR
  AND ex.option_typ = r.OPTION_TYP;
""")

conn.commit()
print("Trades inserted")

Trades inserted


In [10]:
import pandas as pd

pd.read_sql_query("""
SELECT COUNT(*) AS total_rows
FROM raw_fno_data;
""", conn)

,total_rows
0,2533210


In [11]:
pd.read_sql_query("""
WITH oi_calc AS (
    SELECT 
        i.symbol,
        e.exchange_name,
        t.trade_date,
        t.open_int,
        LAG(t.open_int) OVER (
            PARTITION BY i.symbol, e.exchange_name
            ORDER BY t.trade_date
        ) AS prev_oi
    FROM trades t
    JOIN expiries ex ON t.expiry_id = ex.expiry_id
    JOIN instruments i ON ex.instrument_id = i.instrument_id
    JOIN exchanges e ON i.exchange_id = e.exchange_id
)

SELECT 
    symbol,
    exchange_name,
    SUM(open_int - prev_oi) AS total_oi_change
FROM oi_calc
WHERE prev_oi IS NOT NULL
GROUP BY symbol, exchange_name
ORDER BY total_oi_change DESC
LIMIT 10;
""", conn)

,symbol,exchange_name,total_oi_change
0,NIFTYIT,NSE,-13850
1,MRF,NSE,-29740
2,SHREECEM,NSE,-152550
3,BOSCHLTD,NSE,-164220
4,PAGEIND,NSE,-189900
5,OFSS,NSE,-211200
6,NESTLEIND,NSE,-346450
7,EICHERMOT,NSE,-413675
8,TORNTPHARM,NSE,-658000
9,SRF,NSE,-887500


In [12]:
pd.read_sql_query("""
SELECT 
    t.trade_date,
    AVG(t.close_pr) OVER (
        ORDER BY t.trade_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_avg_close
FROM trades t
JOIN expiries ex ON t.expiry_id = ex.expiry_id
JOIN instruments i ON ex.instrument_id = i.instrument_id
WHERE i.symbol = 'NIFTY'
  AND i.instrument_type LIKE 'OPT%'
ORDER BY t.trade_date;
""", conn).head(20)

,trade_date,rolling_avg_close
0,01-AUG-2019,1514.650000
1,01-AUG-2019,1476.650000
2,01-AUG-2019,1482.466667
3,01-AUG-2019,1495.462500
4,01-AUG-2019,1493.270000
5,01-AUG-2019,1496.533333
6,01-AUG-2019,1502.407143
7,01-AUG-2019,1498.564286
8,01-AUG-2019,1431.885714
9,01-AUG-2019,1439.857143


In [13]:
pd.read_sql_query("""
SELECT 
    e.exchange_name,
    i.symbol,
    AVG(t.settle_pr) AS avg_settle_price
FROM trades t
JOIN expiries ex ON t.expiry_id = ex.expiry_id
JOIN instruments i ON ex.instrument_id = i.instrument_id
JOIN exchanges e ON i.exchange_id = e.exchange_id
WHERE 
    (i.symbol LIKE '%GOLD%' AND e.exchange_name = 'MCX')
    OR
    (i.symbol IN ('NIFTY', 'BANKNIFTY') 
     AND i.instrument_type LIKE 'FUT%'
     AND e.exchange_name = 'NSE')
GROUP BY e.exchange_name, i.symbol
ORDER BY e.exchange_name;
""", conn)

,exchange_name,symbol,avg_settle_price
0,NSE,BANKNIFTY,28818.474879
1,NSE,NIFTY,11368.581884


In [14]:
pd.read_sql_query("""
SELECT 
    ex.expiry_dt,
    ex.strike_pr,
    ex.option_typ,
    SUM(t.volume) AS implied_volume
FROM trades t
JOIN expiries ex ON t.expiry_id = ex.expiry_id
JOIN instruments i ON ex.instrument_id = i.instrument_id
WHERE i.instrument_type LIKE 'OPT%'
GROUP BY ex.expiry_dt, ex.strike_pr, ex.option_typ
ORDER BY ex.expiry_dt, ex.strike_pr;
""", conn).head(20)

,expiry_dt,strike_pr,option_typ,implied_volume
0,01-Aug-2019,9600.0,CE,0
1,01-Aug-2019,9600.0,PE,30
2,01-Aug-2019,9650.0,CE,0
3,01-Aug-2019,9650.0,PE,0
4,01-Aug-2019,9700.0,CE,0
5,01-Aug-2019,9700.0,PE,189
6,01-Aug-2019,9750.0,CE,0
7,01-Aug-2019,9750.0,PE,0
8,01-Aug-2019,9800.0,CE,0
9,01-Aug-2019,9800.0,PE,48


In [15]:
pd.read_sql_query("""
SELECT *
FROM (
    SELECT 
        i.symbol,
        t.trade_date,
        t.volume,
        RANK() OVER (
            PARTITION BY i.symbol
            ORDER BY t.volume DESC
        ) AS volume_rank
    FROM trades t
    JOIN expiries ex ON t.expiry_id = ex.expiry_id
    JOIN instruments i ON ex.instrument_id = i.instrument_id
    WHERE t.trade_date >= DATE('now', '-30 day')
)
WHERE volume_rank = 1;
""", conn)

,symbol,trade_date,volume,volume_rank
0,ACC,26-AUG-2019,5163,1
1,ACC,26-AUG-2019,5163,1
2,ACC,26-AUG-2019,5163,1
3,ADANIENT,29-OCT-2019,7526,1
4,ADANIENT,29-OCT-2019,7526,1
...,...,...,...,...
487,YESBANK,31-OCT-2019,215823,1
488,YESBANK,31-OCT-2019,215823,1
489,ZEEL,24-SEP-2019,39376,1
490,ZEEL,24-SEP-2019,39376,1


In [16]:
pd.read_sql_query("""
EXPLAIN QUERY PLAN
SELECT *
FROM trades
WHERE trade_date >= DATE('now', '-30 day');
""", conn)

,id,parent,notused,detail
0,2,0,216,SCAN trades


In [17]:
# Data Loading
# Schema Creation
# Index Creation
# Analytical Queries
# Performance Analysis